# AIC PREPROCESS LAB #
## Basic information ##
- Model name: PE Core Gigantic P14
- Databases: AIC24, AIC25
- Store at: AWS S3 Service
- Vector database: Milvus

**ESSENTIAL INSTALLATION**

**SETUP**

In [ ]:
import requests
import torch
import torch.nn.functional as F
from io import BytesIO
from tqdm import tqdm
from PIL import Image
from open_clip import create_model_from_pretrained

print('Getting model...')
model, preprocess = create_model_from_pretrained('hf-hub:timm/PE-Core-bigG-14-448')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
model = model.eval().to(device)

c:\Users\Bui Thien Nghia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Getting model...
Using device: cuda


CustomTextCLIP(
  (visual): TimmModel(
    (trunk): Eva(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1536, kernel_size=(14, 14), stride=(14, 14), bias=False)
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (rope): RotaryEmbeddingCat()
      (norm_pre): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
      (blocks): ModuleList(
        (0-49): 50 x EvaBlock(
          (norm1): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
          (attn): AttentionRope(
            (qkv): Linear(in_features=1536, out_features=4608, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (proj): Linear(in_features=1536, out_features=1536, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (drop_path1): Identity()
          (norm2): LayerNorm((1536,), eps=1e-05, elementwise_affine

**UTILS FUNCTION DEFINING**

In [25]:
def batch_dataset(dataset, batch_size=128):
  print(f'Batching dataset, batch size: {batch_size}')
  batches = []
  for i in range(0, len(dataset), batch_size):
    batch = dataset[i:i+batch_size] if i < len(dataset) else dataset[i:len(dataset)]
    batches.append(batch)

  print(f'Done patching! Number of batches count: {len(batches)}\n')
  return batches


def preprocess_batch(batch: list[str]):
    images = []
    for link in batch:
        try:
          if 'http' in link:
            link = BytesIO(requests.get(link).content)
          img = Image.open(link)
        except Exception as e:
          print(f'Error: {e}')
          continue

        img_preprocessed = preprocess(img)
        images.append(img_preprocessed)
    
    return torch.stack(images)


def insert_vector_batch(dataset: dict, key_batch: list, vector_batch: list):
  for key, vector in zip(key_batch, vector_batch):
      dataset[key]['vector'] = vector


def tensor_to_vector_list(tensor: torch.Tensor):
    return [row.tolist() for row in tensor]


def compute_batch_features(model, key_batch: list, **options):
  """
  Compute features for all image keys in the batch using given model.

  Arguments:
    - model (open_clip.model.CLIP): CLIP model from OpenCLIP used for feature 
    extraction
    - key_batch (list): list of image keys/path to encode
    **options:
      - preprocessed_batch (list[torch.Tensor]): By default, the function will
      automatically preprocess images for you based on your key_batch. If you
      have a list of preprocessed image, the function will use it instead.

      - auto_insert_into (dict): By default, the function will return the image
      feature batch. If you want to automatically insert vectors into dataset,
      pass your dataset as a dictionary with keys are items in
      key_batch and values are dictionaries with 'vector' key. The code will
      return None then.

  Returns:
    - image_feature_batch (torch.Tensor): Image feature batch tensor with
    dimension (batch, vector_length), or
    - None: if auto_insert_into is True
  """
  with torch.no_grad(), torch.amp.autocast(device):
    if 'preprocessed_batch' in options:
      batch_preprocessed = options['preprocessed_batch']
    else:
      batch_preprocessed = preprocess_batch(key_batch)

    batch_preprocessed = batch_preprocessed.to(device)
    image_feature_batch = model.encode_image(batch_preprocessed, normalize=True)

  # if device == 'cuda':
  #   torch.cuda.empty_cache()

  if 'auto_insert_into' in options:
    insert_vector_batch(options['auto_insert_into'], key_batch, tensor_to_vector_list(image_feature_batch.cpu()))
    return None
  else:
    return image_feature_batch

**SETTING UP DATASET**

In [19]:
dataset = torch.load('aic_2025.pt')
paths = []
for path in list(dataset.keys()):
    paths.append('D:/AIC2025/' + path)

key_batches = batch_dataset(paths, 32)

Batching dataset, batch size: 32
Done patching! Number of batches count: 11947



**COMPUTING & ADDING FEATURE VECTORS TO DATASET**

In [26]:
print(f'Using {device} to compute')
with tqdm(key_batches, desc='Computing batches') as t:
  for batch in t:
    compute_batch_features(model, batch, auto_insert_into=dataset)

    elapsed = t.format_dict['elapsed']
    elapsed_str = t.format_interval(elapsed)

Using cuda to compute


Computing batches:   0%|          | 0/11947 [00:00<?, ?it/s]c:\Users\Bui Thien Nghia\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\linear.py:125: UserWarning: gemm_and_bias error: CUBLAS_STATUS_INTERNAL_ERROR when calling cublasLtMatmul with transpose_mat1 1 transpose_mat2 0 m 1536 n 32 k 1536 mat1_ld 1536 mat2_ld 1536 result_ld 1536 abType 2 cType 2 computeType 68 scaleType 0. Will attempt to recover by calling unfused cublas path. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\cuda\CUDABlas.cpp:1736.)
  return F.linear(input, self.weight, self.bias)
Computing batches:   0%|          | 0/11947 [04:41<?, ?it/s]


AcceleratorError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [25]:
torch.cuda.empty_cache()

**DATA SAVING**

In [14]:
torch.save(dataset, 'aic_2025-wfeat-b2.pt')

**DEBUGGING**

In [1]:
key, value = list(dataset.items())[255]
print(f'Key: {key}\nValue: {value}')

NameError: name 'dataset' is not defined